In [1]:
try:
    import mlflow
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True
    print(f'MLflow OK — {mlflow.get_tracking_uri()}  |  experimento: latam2cloud')
except Exception as e:
    MLFLOW_OK = False
    print(f'MLflow no disponible ({e}) — continuando sin tracking')


MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [2]:

import os, sys, time, gc, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v6.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':         {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta': {'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
}

CHECKPOINT_RF  = True
CHECKPOINT_LGB = True

XGB_PARAMS = {
    'objective':        'multi:softprob',
    'num_class':        4,
    'eval_metric':      'mlogloss',
    'max_depth':        6,
    'learning_rate':    0.05,
    'n_estimators':     500,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'tree_method':      'hist',
    'nthread':          8,
    'random_state':     RANDOM_STATE,
    'verbosity':        0,
}

print(f'XGBoost {xgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')
print(f'XGB_PARAMS: n_estimators={XGB_PARAMS["n_estimators"]}, '
      f'max_depth={XGB_PARAMS["max_depth"]}, lr={XGB_PARAMS["learning_rate"]}')


XGBoost 3.2.0
Input : /srv/work/te_features_v6.parquet
Output: /srv/work/models
XGB_PARAMS: n_estimators=500, max_depth=6, lr=0.05


In [3]:
print('Cargando te_features_v6.parquet (columnas seleccionadas)...')
schema   = pq.read_schema(IN_FEAT)
all_cols = schema.names
keep_cols = [c for c in all_cols if not c.startswith('n_samp')]

df = pq.read_table(IN_FEAT, columns=keep_cols).to_pandas()
print(f'  Shape cargado      : {df.shape}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift  : {df.shape}')

feat_cols = [c for c in df.columns
             if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
             and not c.startswith('n_samp')]
print(f'  Features modelo    : {len(feat_cols)}')

ratio_idx = [i for i, c in enumerate(feat_cols) if 'ratio' in c]
other_idx  = [i for i, c in enumerate(feat_cols) if 'ratio' not in c]

def imputar_nan(X):
    for idx in ratio_idx:
        m = np.isnan(X[:, idx])
        if m.any(): X[m, idx] = 1.0
    for idx in other_idx:
        m = np.isnan(X[:, idx])
        if m.any(): X[m, idx] = 0.0
    return X

train_mask = df[SPLIT_COL] == 'train'
y_train = df.loc[train_mask, 'y'].values.astype(int)

print('\nExtrayendo X_train...')
X_train = imputar_nan(df.loc[train_mask, feat_cols].values.astype('float32'))

del df; gc.collect()
df_val = pq.read_table(IN_FEAT, columns=keep_cols,
                       filters=[('split', '=', 'val')]).to_pandas()
df_val['y'] = df_val.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_val = df_val.dropna(subset=['y'])
y_val  = df_val['y'].values.astype(int)
X_val  = imputar_nan(df_val[feat_cols].values.astype('float32'))
del df_val; gc.collect()

print(f'  X_train: {X_train.shape}  |  y_train Alta: {(y_train==3).sum():,}')
print(f'  X_val  : {X_val.shape}    |  y_val   Alta: {(y_val==3).sum():,}')

sample_weight_train = compute_sample_weight('balanced', y_train)
print(f'  sample_weight rango: [{sample_weight_train.min():.4f}, {sample_weight_train.max():.4f}]')

print('\nDistribución de clases:')
for split_name, y_arr in [('train', y_train), ('val', y_val)]:
    u, c = np.unique(y_arr, return_counts=True)
    print(f'  {split_name}: ' + '  '.join(
        f'{LABEL_COLS[k]}={v:,} ({100*v/len(y_arr):.1f}%)' for k,v in zip(u,c)))


Cargando te_features_v6.parquet (columnas seleccionadas)...
  Shape cargado      : (7628801, 106)
  Post target-shift  : (7626731, 107)
  Features modelo    : 100

Extrayendo X_train...
  X_train: (5349594, 100)  |  y_train Alta: 106,583
  X_val  : (1138777, 100)    |  y_val   Alta: 21,503
  sample_weight rango: [0.2894, 12.5480]

Distribución de clases:
  train: Normal=4,621,399 (86.4%)  Baja=509,392 (9.5%)  Media=112,220 (2.1%)  Alta=106,583 (2.0%)
  val: Normal=1,024,150 (89.9%)  Baja=70,518 (6.2%)  Media=22,606 (2.0%)  Alta=21,503 (1.9%)


In [4]:
def evaluar(nombre, y_true, proba):
    y_pred = np.argmax(proba, axis=1)
    qwk  = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    rec  = (y_pred[y_true==3] == 3).mean() if (y_true==3).any() else 0.0
    mf1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac  = balanced_accuracy_score(y_true, y_pred)
    cand = qwk >= UMBRALES['qwk']['floor'] and rec >= UMBRALES['recall_alta']['floor']
    print(f'{"─"*80}')
    print(f'  {nombre}')
    print(f'{"─"*80}')
    print(f'  QWK      = {qwk:.4f}  {"✅" if qwk >= UMBRALES["qwk"]["floor"] else "❌"}  '
          f'(floor≥{UMBRALES["qwk"]["floor"]})')
    print(f'  RecAlta  = {rec:.4f}  {"✅" if rec >= UMBRALES["recall_alta"]["floor"] else "❌"}  '
          f'(floor≥{UMBRALES["recall_alta"]["floor"]})')
    print(f'  MacroF1  = {mf1:.4f}')
    print(f'  BalAcc   = {bac:.4f}')
    print(f'  CANDIDATO: {"✅ SÍ" if cand else "❌ NO"}')
    cm = confusion_matrix(y_true, y_pred)
    print(f'  Confusion matrix (Normal/Baja/Media/Alta):')
    for i, (lab, row) in enumerate(zip(LABEL_COLS.values(), cm)):
        print(f'    {lab:<7}: {list(row)}')
    return {'qwk': qwk, 'recall_alta': rec, 'macro_f1': mf1,
            'bal_acc': bac, 'candidato': cand, 'nombre': nombre}

print('evaluar() definida.')


evaluar() definida.


In [5]:
print('Baseline estático — etiqueta actual como predicción...')
n = len(y_val)
proba_base = np.zeros((n, 4), dtype='float32')
proba_base[np.arange(n), y_val] = 1.0
res_base = evaluar('Baseline estático (etiqueta actual — sin ML)', y_val, proba_base)
del proba_base


Baseline estático — etiqueta actual como predicción...
────────────────────────────────────────────────────────────────────────────────
  Baseline estático (etiqueta actual — sin ML)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 1.0000  ✅  (floor≥0.65)
  RecAlta  = 1.0000  ✅  (floor≥0.55)
  MacroF1  = 1.0000
  BalAcc   = 1.0000
  CANDIDATO: ✅ SÍ
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal : [1024150, 0, 0, 0]
    Baja   : [0, 70518, 0, 0]
    Media  : [0, 0, 22606, 0]
    Alta   : [0, 0, 0, 21503]


In [6]:

print(f'XGBoost v12 — n_estimators={XGB_PARAMS["n_estimators"]}, '
      f'max_depth={XGB_PARAMS["max_depth"]}, lr={XGB_PARAMS["learning_rate"]}')
print(f'  X_train: {X_train.shape}  |  y_train Alta: {(y_train==3).sum():,}')
print(f'  Tiempo estimado: ~90-150 min (5.35M filas, hist method, 8 threads)')

t0 = time.time()
xgb_v12 = xgb.XGBClassifier(**XGB_PARAMS)
xgb_v12.fit(X_train, y_train, sample_weight=sample_weight_train,
            eval_set=[(X_val, y_val)], verbose=100)
elapsed = (time.time() - t0) / 60
print(f'  Entrenado en {elapsed:.1f} min')

t1 = time.time()
proba_xgb_v12 = xgb_v12.predict_proba(X_val).astype('float32')
print(f'  Predicción val: {(time.time()-t1):.1f}s')

np.save(f'{OUT_DIR}/xgb_v12_val_proba.npy', proba_xgb_v12)
print(f'  Guardado: xgb_v12_val_proba.npy')

xgb_v12.save_model(f'{OUT_DIR}/xgb_v12_model.ubj')
print(f'  Guardado: xgb_v12_model.ubj')

fi_values_xgb = xgb_v12.feature_importances_.copy()
del xgb_v12; gc.collect()
print(f'  xgb_v12 liberado de RAM — fi_values_xgb conservado')

res_xgb_v12 = evaluar('XGBoost v12 (max_depth=6, lr=0.05, 500 est)', y_val, proba_xgb_v12)

if MLFLOW_OK:
    with mlflow.start_run(run_name='xgb_v12'):
        mlflow.log_params({k: v for k, v in XGB_PARAMS.items()
                           if k not in ('objective', 'eval_metric', 'tree_method')})
        mlflow.log_params({'dataset': 'te_features_v6.parquet', 'checkpoint': False})
        mlflow.log_metrics({'qwk_val': res_xgb_v12['qwk'],
                            'recall_alta_val': res_xgb_v12['recall_alta']})
    print('  MLflow run xgb_v12 registrado')

del X_train, sample_weight_train; gc.collect()
print('  X_train liberado (Fix-MEM-04)')


XGBoost v12 — n_estimators=500, max_depth=6, lr=0.05
  X_train: (5349594, 100)  |  y_train Alta: 106,583
  Tiempo estimado: ~90-150 min (5.35M filas, hist method, 8 threads)
[0]	validation_0-mlogloss:1.33216
[100]	validation_0-mlogloss:0.49261
[200]	validation_0-mlogloss:0.48986
[300]	validation_0-mlogloss:0.48736
[400]	validation_0-mlogloss:0.48182
[499]	validation_0-mlogloss:0.47863
  Entrenado en 12.8 min
  Predicción val: 9.1s
  Guardado: xgb_v12_val_proba.npy
  Guardado: xgb_v12_model.ubj
  xgb_v12 liberado de RAM — fi_values_xgb conservado
────────────────────────────────────────────────────────────────────────────────
  XGBoost v12 (max_depth=6, lr=0.05, 500 est)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.3684  ❌  (floor≥0.65)
  RecAlta  = 0.7764  ✅  (floor≥0.55)
  MacroF1  = 0.5347
  BalAcc   = 0.7579
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal : [863997, 50323, 42419, 67411]
    Baja   : [6173, 

In [7]:

print('Cargando checkpoints RF y LGB desde v10...')

if CHECKPOINT_RF:
    proba_rf_v10 = np.load(f'{OUT_DIR}/rf_v10_val_proba.npy')
    assert proba_rf_v10.shape[0] == len(y_val), (
        f'Mismatch RF: {proba_rf_v10.shape[0]} vs {len(y_val)}')
    print(f'  proba_rf_v10  : {proba_rf_v10.shape} ✅')
    res_rf = evaluar('RF v10 [checkpoint] (referencia v12)', y_val, proba_rf_v10)
else:
    proba_rf_v10 = None
    print('  RF checkpoint desactivado')

if CHECKPOINT_LGB:
    proba_lgb_v10 = np.load(f'{OUT_DIR}/lgb_v10_val_proba.npy')
    assert proba_lgb_v10.shape[0] == len(y_val), (
        f'Mismatch LGB: {proba_lgb_v10.shape[0]} vs {len(y_val)}')
    print(f'  proba_lgb_v10 : {proba_lgb_v10.shape} ✅')
    res_lgb = evaluar('LGB v10 [checkpoint] (referencia v12)', y_val, proba_lgb_v10)
else:
    proba_lgb_v10 = None
    print('  LGB checkpoint desactivado')


Cargando checkpoints RF y LGB desde v10...
  proba_rf_v10  : (1138777, 4) ✅
────────────────────────────────────────────────────────────────────────────────
  RF v10 [checkpoint] (referencia v12)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.6223  ❌  (floor≥0.65)
  RecAlta  = 0.5073  ❌  (floor≥0.55)
  MacroF1  = 0.6500
  BalAcc   = 0.6434
  CANDIDATO: ❌ NO
  Confusion matrix (Normal/Baja/Media/Alta):
    Normal : [996468, 15240, 3730, 8712]
    Baja   : [15776, 50796, 2738, 1208]
    Media  : [7676, 2508, 8436, 3986]
    Alta   : [8364, 540, 1691, 10908]
  proba_lgb_v10 : (1138777, 4) ✅
────────────────────────────────────────────────────────────────────────────────
  LGB v10 [checkpoint] (referencia v12)
────────────────────────────────────────────────────────────────────────────────
  QWK      = 0.5740  ❌  (floor≥0.65)
  RecAlta  = 0.6487  ✅  (floor≥0.55)
  MacroF1  = 0.6375
  BalAcc   = 0.7127
  CANDIDATO: ❌ NO
  Confusion matrix (No

In [8]:

print('Grid search pesos ensemble XGB / RF / LGB...')
print('  Espacio: w_XGB + w_RF + w_LGB = 1.0, paso 0.05')

best_cand   = None
best_qwk    = -1.0
best_nocand = None

results = []
step = 0.05
vals = [round(i * step, 2) for i in range(int(1/step) + 1)]

for w_xgb in vals:
    for w_rf in vals:
        w_lgb = round(1.0 - w_xgb - w_rf, 2)
        if w_lgb < 0 or abs(w_xgb + w_rf + w_lgb - 1.0) > 1e-6:
            continue
        if w_xgb < 0.50:
            continue

        proba = (w_xgb * proba_xgb_v12
                 + (w_rf  * proba_rf_v10  if proba_rf_v10  is not None else 0)
                 + (w_lgb * proba_lgb_v10 if proba_lgb_v10 is not None else 0))

        y_pred = np.argmax(proba, axis=1)
        qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')
        rec = (y_pred[y_val==3] == 3).mean() if (y_val==3).any() else 0.0
        cand = qwk >= UMBRALES['qwk']['floor'] and rec >= UMBRALES['recall_alta']['floor']
        results.append((w_xgb, w_rf, w_lgb, qwk, rec, cand))

        if cand and qwk > best_qwk:
            best_qwk  = qwk
            best_cand = (w_xgb, w_rf, w_lgb, qwk, rec, proba.copy())
        if not cand and (best_nocand is None or qwk > best_nocand[3]):
            best_nocand = (w_xgb, w_rf, w_lgb, qwk, rec, proba.copy())

results.sort(key=lambda x: -x[3])
print(f'\n  {"w_XGB":>6}  {"w_RF":>5}  {"w_LGB":>6}     {"QWK":>6}    {"RecAlta":>7}  CAND')
print(f'  {"─"*56}')
for row in results[:10]:
    w_x, w_r, w_l, q, r, c = row
    marker = '✅' if c else ''
    print(f'    {w_x:.2f}    {w_r:.2f}    {w_l:.2f}   {q:.4f}     {r:.4f}    {marker}')

if best_cand:
    w_x, w_r, w_l, q, r, proba_best = best_cand
    print(f'\n  ✅ CANDIDATO: XGB×{w_x}+RF×{w_r}+LGB×{w_l}  QWK={q:.4f}  RecAlta={r:.4f}')
    proba_ensemble_v12 = proba_best
    best_weights = (w_x, w_r, w_l)
else:
    w_x, w_r, w_l, q, r, proba_best = best_nocand
    print(f'\n  Sin combinación candidata — mejor por QWK: XGB×{w_x}+RF×{w_r}+LGB×{w_l}')
    print(f'  QWK={q:.4f}  RecAlta={r:.4f}')
    proba_ensemble_v12 = proba_best
    best_weights = (w_x, w_r, w_l)

w_xgb_best, w_rf_best, w_lgb_best = best_weights
res_ens_v12 = evaluar(
    f'Ensemble v12 óptimo (XGB×{w_xgb_best}+RF×{w_rf_best}+LGB×{w_lgb_best})',
    y_val, proba_ensemble_v12)


Grid search pesos ensemble XGB / RF / LGB...
  Espacio: w_XGB + w_RF + w_LGB = 1.0, paso 0.05

   w_XGB   w_RF   w_LGB        QWK    RecAlta  CAND
  ────────────────────────────────────────────────────────
    0.50    0.50    0.00   0.5410     0.7048    
    0.50    0.45    0.05   0.5346     0.7107    
    0.50    0.40    0.10   0.5292     0.7159    
    0.50    0.35    0.15   0.5237     0.7195    
    0.55    0.45    -0.00   0.5205     0.7232    
    0.50    0.30    0.20   0.5186     0.7220    
    0.55    0.40    0.05   0.5151     0.7266    
    0.50    0.25    0.25   0.5135     0.7239    
    0.55    0.35    0.10   0.5102     0.7294    
    0.50    0.20    0.30   0.5089     0.7258    

  Sin combinación candidata — mejor por QWK: XGB×0.5+RF×0.5+LGB×0.0
  QWK=0.5410  RecAlta=0.7048
────────────────────────────────────────────────────────────────────────────────
  Ensemble v12 óptimo (XGB×0.5+RF×0.5+LGB×0.0)
─────────────────────────────────────────────────────────────────────────────

In [9]:

import pandas as pd

fi_df = pd.DataFrame({
    'feature': feat_cols,
    'importance': fi_values_xgb
}).sort_values('importance', ascending=False).reset_index(drop=True)

fi_df['pct_acum'] = (fi_df['importance'].cumsum() / fi_df['importance'].sum() * 100).round(1)

print(f'Top 20 features por gain (XGBoost v12):')
print(f'  {"Rank":>4}  {"Feature":<45}  {"Gain":>8}   {"% acum":>7}')
print(f'  {"─"*70}')
for i, row in fi_df.head(20).iterrows():
    print(f'  {i+1:>4}  {row.feature:<45}  {row.importance:.5f}   {row.pct_acum:>6.1f}%')

print(f'\n  Suma top-20: {fi_df.head(20).importance.sum():.4f} '
      f'({fi_df.head(20).pct_acum.iloc[-1]:.1f}% de importancia total)')

fi_df.head(20).to_csv(f'{OUT_DIR}/xgb_v12_feature_importance_top20.csv', index=False)
print(f'  Guardado: xgb_v12_feature_importance_top20.csv')


Top 20 features por gain (XGBoost v12):
  Rank  Feature                                            Gain    % acum
  ──────────────────────────────────────────────────────────────────────
     1  y_lag1                                         0.31144     31.1%
     2  y_lag2                                         0.14520     45.7%
     3  y_lag3                                         0.07464     53.1%
     4  mean_loss                                      0.05480     58.6%
     5  lag1_mean_loss                                 0.03060     61.7%
     6  streak_length                                  0.02935     64.6%
     7  lag2_mean_loss                                 0.02084     66.7%
     8  p95_loss                                       0.01710     68.4%
     9  std_lat                                        0.01631     70.0%
    10  lag1_delta_p95_loss                            0.01603     71.6%
    11  delta_p95_loss                                 0.01531     73.2%
    12  la

In [11]:

np.save(f'{OUT_DIR}/ens_v12_val_proba.npy', proba_ensemble_v12)
print(f'ens_v12_val_proba.npy guardado')

candidato_str = (f'XGB×{w_xgb_best}+RF×{w_rf_best}+LGB×{w_lgb_best}'
                 f'  QWK={res_ens_v12["qwk"]:.4f}'
                 f'  RecAlta={res_ens_v12["recall_alta"]:.4f}')
print(f'Ensemble v12 mejor: {candidato_str}')

if MLFLOW_OK:
    with mlflow.start_run(run_name='ensemble_v12'):
        mlflow.log_params({'w_xgb': w_xgb_best, 'w_rf': w_rf_best, 'w_lgb': w_lgb_best,
                           'dataset': 'te_features_v6.parquet'})
        mlflow.log_metrics({'qwk_val': res_ens_v12['qwk'],
                            'recall_alta_val': res_ens_v12['recall_alta']})
    print('MLflow run ensemble_v12 registrado')

print('\n' + '='*100)
print('  TABLA COMPARATIVA v9 → v10 → v11 → v12 (validación)')
print('='*100)
rows = [
    ('Baseline estático v12',                        res_base['qwk'],    res_base['recall_alta'],    res_base['macro_f1'],    '❌'),
    ('XGBoost v12 (max_depth=6, lr=0.05, 500 est)', res_xgb_v12['qwk'], res_xgb_v12['recall_alta'], res_xgb_v12['macro_f1'], '✅' if res_xgb_v12['candidato'] else '❌'),
    ('RF v10 [ckpt]',                                res_rf['qwk'],      res_rf['recall_alta'],      res_rf['macro_f1'],      '❌'),
    ('LGB v10 [ckpt]',                               res_lgb['qwk'],     res_lgb['recall_alta'],     res_lgb['macro_f1'],     '❌'),
    (f'Ensemble v12 ({w_xgb_best}×XGB+{w_rf_best}×RF+{w_lgb_best}×LGB)', res_ens_v12['qwk'], res_ens_v12['recall_alta'], res_ens_v12['macro_f1'], '✅' if res_ens_v12['candidato'] else '❌'),
]
print(f'  {"Modelo":<52}  {"QWK":>7}    {"RecAlta":>7}   {"MacroF1":>8}   Cand')
print(f'  {"─"*90}')
for nombre, qwk, rec, mf1, cand in rows:
    print(f'  {nombre:<52}  {qwk:.4f}     {rec:.4f}    {mf1:.4f}      {cand}')
print(f'  {"─"*90}')
print(f'  {"── REF v9 (83d, v5) ── Ens D38":<52}  {"0.6529":>7}     {"0.6509":>7}        {"──":>8}      ✅')
print(f'  {"── REF v10 (162d, v6) ── Ens óptimo":<52}  {"0.6227":>7}     {"0.5237":>7}        {"──":>8}      ❌')
print(f'  {"── REF v11 (83d, v6/P1) ── Ens óptimo":<52}  {"0.6231":>7}     {"0.5254":>7}        {"──":>8}      ❌')
print()
print(f'  Umbral candidato: QWK≥0.65 AND RecAlta≥0.55  (Benchmarking Criterios Éxito v2.0)')
cand_final = res_ens_v12["candidato"] or res_xgb_v12["candidato"]
print(f'  CANDIDATO FINAL v12: {"✅ " + candidato_str if cand_final else "❌ ninguno — ver LOG-46/D46-5"}')
if cand_final:
    print(f'  → Próximo paso: 06 evaluacion test set v7.ipynb (ONE-SHOT — Sep 8-30)')
else:
    print(f'  → Proceder con v9 Ensemble D38 (LOG-25) como candidato final para OE2')


ens_v12_val_proba.npy guardado
Ensemble v12 mejor: XGB×0.5+RF×0.5+LGB×0.0  QWK=0.5410  RecAlta=0.7048
🏃 View run ensemble_v12 at: http://localhost:6000/#/experiments/681062908553326840/runs/5f921852479143d88eb05262961194d1
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow run ensemble_v12 registrado

  TABLA COMPARATIVA v9 → v10 → v11 → v12 (validación)
  Modelo                                                    QWK    RecAlta    MacroF1   Cand
  ──────────────────────────────────────────────────────────────────────────────────────────
  Baseline estático v12                                 1.0000     1.0000    1.0000      ❌
  XGBoost v12 (max_depth=6, lr=0.05, 500 est)           0.3684     0.7764    0.5347      ❌
  RF v10 [ckpt]                                         0.6223     0.5073    0.6500      ❌
  LGB v10 [ckpt]                                        0.5740     0.6487    0.6375      ❌
  Ensemble v12 (0.5×XGB+0.5×RF+0.0×LGB)                 0.54